# Stage 9, observe: an operations agent over Airflow and the Spark History Server

Northwind Gear's daily pipeline runs as the Airflow DAG **`northwind_daily`** (`ops/dags/northwind_daily.py`, stage 8):
it appends the day's order events to `shop_bronze.order_events` over Spark Connect, runs `dbt build`, checks the data,
publishes `shop_gold.product_kpis` and waits until the semantic model is `Published` again. The Spark notebooks of stages 1–4 leave their applications in the Spark History Server.

Instead of clicking through two UIs, this notebook gives an agent (built on [AWS Strands](https://strandsagents.com), `lab_agents.py`) the tools of two MCP servers the chart runs:

* **Airflow MCP** (Astronomer `astro-airflow-mcp`, read-only, logged in as its own `viewer` user): DAGs, runs, task
  instances, logs.
* **Spark History MCP** (Kubeflow `mcp-apache-spark-history-server`): applications, jobs, stages, executors.

Helm chart only (`airflowMcp.enabled`, `sparkHistoryMcp.enabled`; the workstation profile turns both on, and mounts
`ops/dags` as Airflow's DAG folder, so the DAG is already deployed and unpaused). Outside the lab's Jupyter the first cell
stops with a message.

Without an LLM (`llm.provider=none`) the tools-by-hand cells run and the agent cells print `skipped (no LLM configured)`.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import lab_utils as lu

lu.require_ops()
if lu.llm_available():
    import lab_agents as la
    lu.require_tool_model()
for server, url in lu.ops_servers().items():
    names = sorted(t["name"] for t in lu.ops_tools(server))
    print(f"{server} ({url}): {len(names)} tools\n   {', '.join(names)}\n")

## 1. One tool call by hand
Every MCP tool is a function with a JSON schema. Before handing them to the LLM, call one directly: the Spark
applications the History Server knows about (the notebooks' Spark Connect sessions and the SparkApplications).

In [ ]:
spark_tools = {t["name"] for t in lu.ops_tools("spark_history")} if "spark_history" in lu.ops_servers() else set()
if "list_applications" in spark_tools:
    apps = lu.ops_call("spark_history", "list_applications", {})
    print(apps[:1500] if isinstance(apps, str) else json.dumps(apps, indent=1, default=str)[:1500])
else:
    print("no list_applications tool here; the agent below picks from the tools listed above")

## 2. Ask the agent
`lab_agents.ops_agent()` is a Strands agent offered the read-only tools of both servers (`lab_utils.OPS_TOOLS`: nothing
that can trigger, pause or delete). It lists first, then drills down. Each step is shown below the answer, so you can
check that the answer comes from the tools. Today's date is in its prompt: a model otherwise filters on a training-era date.

In [ ]:
def investigate(question, agent):
    out = la.ask(agent, question)
    print(f"Q: {question}\n\nA: {out['answer']}\n")
    return pd.DataFrame([{"tool": s["tool"], "args": json.dumps(s["args"]),
                          "result": json.dumps(s.get("result"), default=str)[:300]} for s in out["steps"]])

if lu.skip_unless_llm("the operations agent"):
    ops = la.ops_agent()
    agent = ops.__enter__()      # one agent for both questions (closed after the second)
    display(investigate("Did the northwind_daily DAG run? Give the state of its latest run and of each task in it. "
                        "If a task failed, say why from its log.", agent))

In [ ]:
if lu.llm_available():
    display(investigate("Which Spark applications ran most recently, and which of them took longest? Name the application ids.", agent))
    ops.__exit__(None, None, None)

## What to look for
* The first answer names the latest `northwind_daily` run and its tasks (`ingest`, `build`, `checks`, `gold`,
  `published`). If one failed, its log says why, e.g. a failing dbt test in `build`; fix it, then trigger the DAG
  again in the Airflow UI (`http://airflow.datalab.localhost:8080` on the workstation).
* The Airflow MCP server is **read-only** and logs in as its own least-privilege user: the agent can look, not
  trigger or change anything.
* Small models sometimes skip a step or misread an id; the steps table is how you check. A bigger tool model
  (`llm.toolModel`) investigates more thoroughly.